# BERTopic inspection (publication month 2026-01)

Evidence for selected works published 2026-01-01 through 2026-01-31 on `demo_vml`: embedding coverage in Qdrant, work summaries, and discovery-topic headlines/labels.

A discovery topic label (headline) is not a weak signal by itself. It only names a candidate cluster for later weak-signal assessment (WISDOM TEM and eligibility). This notebook shows live counts and a few **succeeded** examples so a reader can judge whether completed labels look usable as weak-signal *candidates*: topic size, outlier share, headline specificity, and whether member summaries support the headline.

Label rows may be `succeeded` or `failed`. Failed rows have no headline. Ops confirmed the failures are HTTP 403 `Project not found` from the labeling API (missing/unbound project), not a sampling bug. This notebook does **not** call the labeling API; it only reads persisted rows.

Queries, not hardcoded counts. Re-run top to bottom after more labels land.


In [1]:
import json
import os
import sys
import time
from datetime import date
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir():
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
from IPython.display import display
from qdrant_client import QdrantClient
from qdrant_client.http import models as qm
from sqlalchemy import and_, func, inspect, or_, select, text

from src.common.db import create_processor_engine
from src.common.settings import Settings
from src.corpus.select import _reconstruct_abstract
from src.models.labels import TopicLabel, WorkSummary
from src.models.topics import DiscoveryTopic, TopicRun, WorkTopicAssignment
from src.models.upstream import work_current, work_versions

TOK_RATIO = 3.0
CONTEXT_LEN = 4096

settings = Settings()
engine = create_processor_engine(
    settings.database_url, upstream_schema=settings.upstream_schema
)
qdrant = QdrantClient(url=settings.qdrant_url)
config = settings.corpus.model_copy(
    update={
        "published_from": date(2026, 1, 1),
        "published_to": date(2026, 1, 31),
    }
)

pd.set_option("display.max_colwidth", 160)
print(f"Published window: {config.published_from} to {config.published_to}")
print(f"Qdrant collection: {settings.qdrant_collection}")


Published window: 2026-01-01 to 2026-01-31
Qdrant collection: openalex_tabstract


## Works

Selected works use the corpus filter from settings: collection scopes, work types, publication interval, excluded OpenAlex domains, a stored abstract, and no cross-scope version conflict.


In [2]:
LOAD_CACHE_WORKS_DF = True
works_cache_path = Path("data") / "works_df.parquet"
entity_ids_cache_path = Path("data") / "works_entity_ids_2026-01.parquet"

print(
    f"Scopes: {len(config.scope_ids)}. "
    f"\nTypes: {', '.join(config.work_types)}. "
    f"\nPublished: {config.published_from.isoformat()} to {config.published_to.isoformat()}. "
    f"\nExcluded domains: {', '.join(config.excluded_domain_ids) or 'none'}."
)

payload = work_versions.c.payload
work_type = payload["type"].astext
published = payload["publication_date"].astext
domain_id = payload["primary_topic"]["domain"]["id"].astext
abstract_index = payload["abstract_inverted_index"]
abstract_kind = func.jsonb_typeof(abstract_index)
joined = work_current.join(
    work_versions, work_versions.c.id == work_current.c.version_id
)
empty_object = text("'{}'::jsonb")
empty_string_object = text("'\"{}\"'::jsonb")
null_string_object = text("'\"null\"'::jsonb")
empty_quoted = text("'\"\"'::jsonb")
selected_filter = [
    work_current.c.scope_id.in_(config.scope_ids),
    work_type.in_(config.work_types),
    published >= config.published_from.isoformat(),
    published <= config.published_to.isoformat(),
    or_(domain_id.is_(None), domain_id.not_in(config.excluded_domain_ids)),
    or_(
        and_(abstract_kind == "object", abstract_index != empty_object),
        and_(
            abstract_kind == "string",
            abstract_index != empty_string_object,
            abstract_index != null_string_object,
            abstract_index != empty_quoted,
        ),
    ),
]
if len(config.scope_ids) > 1:
    versions = func.count(func.distinct(work_current.c.version_id))
    conflicts = (
        select(work_current.c.entity_id)
        .where(work_current.c.scope_id.in_(config.scope_ids))
        .group_by(work_current.c.entity_id)
        .having(versions > 1)
    )
    selected_filter.append(work_current.c.entity_id.not_in(conflicts))

title_text = payload["title"].astext
title_length = func.length(func.trim(title_text))
selected = (
    select(
        work_type.label("type"),
        title_length.label("title_length"),
        abstract_index.label("abstract_index"),
    )
    .select_from(joined)
    .where(*selected_filter)
)
entity_id_query = (
    select(work_current.c.entity_id.label("work_id"))
    .select_from(joined)
    .where(*selected_filter)
)

if LOAD_CACHE_WORKS_DF and works_cache_path.is_file():
    works_df = pd.read_parquet(works_cache_path)
    works_by_type = (
        works_df["type"].value_counts().rename_axis("type").reset_index(name="works")
    )
    print(f"Loaded works from {works_cache_path}")
elif not config.scope_ids:
    works_df = pd.DataFrame(columns=["type", "title_length", "abstract_length"])
    works_by_type = pd.DataFrame(columns=["type", "works"])
    print("No collection scopes are configured. Set PWF_CORPUS__SCOPE_IDS.")
else:
    with engine.connect() as connection:
        started = time.perf_counter()
        connection.execute(text("SET max_parallel_workers_per_gather = 0"))
        works_df = pd.read_sql(selected, connection)
        elapsed = time.perf_counter() - started
    print(f"Query duration: {elapsed:.2f} s")
    works_df["title_length"] = works_df["title_length"].fillna(0).astype(int)
    works_df["abstract_length"] = [
        len(_reconstruct_abstract(idx) or "") for idx in works_df["abstract_index"]
    ]
    works_df["abstract_index"] = [
        value if isinstance(value, str) else json.dumps(value)
        for value in works_df["abstract_index"]
    ]
    works_by_type = (
        works_df["type"].value_counts().rename_axis("type").reset_index(name="works")
    )
    works_cache_path.parent.mkdir(parents=True, exist_ok=True)
    works_df.to_parquet(works_cache_path, index=False)
    print(f"Saved works to {works_cache_path}")

if entity_ids_cache_path.is_file():
    entity_ids_df = pd.read_parquet(entity_ids_cache_path)
    print(f"Loaded entity ids from {entity_ids_cache_path}")
elif not config.scope_ids:
    entity_ids_df = pd.DataFrame(columns=["work_id"])
else:
    with engine.connect() as connection:
        started = time.perf_counter()
        connection.execute(text("SET max_parallel_workers_per_gather = 0"))
        entity_ids_df = pd.read_sql(entity_id_query, connection)
        elapsed = time.perf_counter() - started
    print(f"Entity-id query duration: {elapsed:.2f} s")
    entity_ids_cache_path.parent.mkdir(parents=True, exist_ok=True)
    entity_ids_df.to_parquet(entity_ids_cache_path, index=False)
    print(f"Saved entity ids to {entity_ids_cache_path}")

selected_works = len(works_df)
selected_work_ids = set(entity_ids_df["work_id"].astype(str))
print(f"Selected works: {selected_works}")
print(f"Selected work ids: {len(selected_work_ids)}")
display(works_by_type)


Scopes: 1. 
Types: article, preprint, conference-paper. 
Published: 2026-01-01 to 2026-01-31. 
Excluded domains: https://openalex.org/domains/2.
Loaded works from data/works_df.parquet
Loaded entity ids from data/works_entity_ids_2026-01.parquet
Selected works: 38534
Selected work ids: 38534


,type,works
0,article,33590
1,conference-paper,2880
2,preprint,2064


In [3]:
works_df.head(3)

Out[0]: 
               type  ...  abstract_length
0  conference-paper  ...             1195
1           article  ...             2786
2           article  ...             1796

[3 rows x 4 columns]


## Embeddings

Processor Qdrant collection from settings (`PWF_QDRANT_COLLECTION`), collection point total, points whose payload `publication_date` falls in 2026-01, and a small sample of point ids aligned to work ids.


In [4]:
collection = settings.qdrant_collection
print(f"Collection: {collection}")
print(
    "Queried: Qdrant collection "
    f"`{collection}` with payload publication_date in "
    f"[{config.published_from}, {config.published_to}]"
)

if not qdrant.collection_exists(collection):
    collection_total = 0
    embedded_work_ids = set()
    point_samples = []
    print("Collection is not present.")
else:
    collection_total = qdrant.count(collection, exact=True).count
    month_filter = qm.Filter(
        must=[
            qm.FieldCondition(
                key="publication_date",
                range=qm.DatetimeRange(
                    gte=config.published_from.isoformat(),
                    lte=config.published_to.isoformat(),
                ),
            )
        ]
    )
    month_points = qdrant.count(collection, count_filter=month_filter, exact=True).count
    print(f"Collection point total: {collection_total}")
    print(f"Points with publication_date in month: {month_points}")

    embedded_work_ids = set()
    point_samples = []
    offset = None
    started = time.perf_counter()
    while True:
        points, offset = qdrant.scroll(
            collection_name=collection,
            scroll_filter=month_filter,
            limit=256,
            offset=offset,
            with_payload=["work_id"],
            with_vectors=False,
        )
        for point in points:
            work_id = (point.payload or {}).get("work_id")
            if work_id:
                embedded_work_ids.add(str(work_id))
            if len(point_samples) < 8:
                point_samples.append({"point_id": point.id, "work_id": work_id})
        if offset is None:
            break
    print(f"Scroll duration: {time.perf_counter() - started:.2f} s")

selected_with_vectors = len(selected_work_ids & embedded_work_ids)
print(f"Selected works with vectors: {selected_with_vectors}")
print(f"Selected works missing vectors: {len(selected_work_ids - embedded_work_ids)}")
sample_frame = pd.DataFrame(point_samples)
print("Sample point ids / work ids:")
display(sample_frame)


Collection: openalex_tabstract
Queried: Qdrant collection `openalex_tabstract` with payload publication_date in [2026-01-01, 2026-01-31]
Collection point total: 38030
Points with publication_date in month: 38030
Scroll duration: 43.66 s
Selected works with vectors: 38030
Selected works missing vectors: 504
Sample point ids / work ids:


,point_id,work_id
0,00005099-6235-5b34-bcdb-27da35f18be7,https://openalex.org/W7125911450
1,0002281d-1e98-5525-8c95-af1998991b8e,https://openalex.org/W7128804491
2,0003d573-0cb4-554f-8bb0-fefc99afde05,https://openalex.org/W7125812079
3,00050616-5438-5c85-8bf2-1cad21805505,https://openalex.org/W7122701870
4,00051202-5bdd-54ba-9ea1-7c89101caf73,https://openalex.org/W7118159315
5,000735a1-cebb-5925-ba03-ed71c5756dfe,https://openalex.org/W7125813281
6,000baa8d-de5f-58c3-ba60-91eff9ec1487,https://openalex.org/W7124286321
7,000c1a9a-95e0-58c5-9349-9d679c4c5162,https://openalex.org/W7151264759


## Topic run for this corpus

Latest row in `pwf_topic_runs` whose publication interval covers 2026-01 (prefer exact match on the notebook window). Report run status, discovery topic count, and outlier count from the run row and from `pwf_work_topic_assignments` (`is_outlier` / `bertopic_topic_id = -1`).


In [5]:
inspector = inspect(engine)
topic_run = None
topic_run_id = None
discovery_topic_count = 0
outlier_assignment_count = 0
outlier_bertopic_count = 0

if not inspector.has_table(TopicRun.__tablename__):
    print(f"Table queried: {TopicRun.__tablename__} (missing)")
    print("Topic runs covering this corpus: 0")
else:
    print(f"Table queried: {TopicRun.__tablename__}")
    covering = (
        select(TopicRun)
        .where(
            TopicRun.published_from <= config.published_from,
            TopicRun.published_to >= config.published_to,
        )
        .order_by(TopicRun.started_at.desc())
    )
    exact = (
        select(TopicRun)
        .where(
            TopicRun.published_from == config.published_from,
            TopicRun.published_to == config.published_to,
        )
        .order_by(TopicRun.started_at.desc())
    )
    with engine.connect() as connection:
        exact_rows = connection.execute(exact).mappings().all()
        covering_rows = connection.execute(covering).mappings().all()
        run_total = connection.execute(
            select(func.count()).select_from(TopicRun)
        ).scalar_one()
    print(f"All topic runs in database: {run_total}")
    print(
        f"Runs with exact window {config.published_from}..{config.published_to}: "
        f"{len(exact_rows)}"
    )
    print(f"Runs covering the window: {len(covering_rows)}")
    chosen = exact_rows[0] if exact_rows else (covering_rows[0] if covering_rows else None)
    if chosen is None:
        print("No topic run covers this corpus. Downstream counts use run_id=None.")
        display(
            pd.DataFrame(
                columns=[
                    "run_id",
                    "status",
                    "published_from",
                    "published_to",
                    "work_count",
                    "topic_count",
                    "outlier_count",
                    "outlier_rate",
                    "started_at",
                ]
            )
        )
    else:
        topic_run = dict(chosen)
        topic_run_id = topic_run["run_id"]
        print(f"Selected topic_run_id: {topic_run_id}")
        display(
            pd.DataFrame(
                [
                    {
                        "run_id": topic_run["run_id"],
                        "status": topic_run["status"],
                        "published_from": topic_run["published_from"],
                        "published_to": topic_run["published_to"],
                        "work_count": topic_run["work_count"],
                        "topic_count": topic_run["topic_count"],
                        "outlier_count": topic_run.get("outlier_count"),
                        "outlier_rate": topic_run.get("outlier_rate"),
                        "started_at": topic_run["started_at"],
                        "finished_at": topic_run.get("finished_at"),
                    }
                ]
            )
        )
        if inspector.has_table(DiscoveryTopic.__tablename__):
            with engine.connect() as connection:
                discovery_topic_count = connection.execute(
                    select(func.count())
                    .select_from(DiscoveryTopic)
                    .where(DiscoveryTopic.topic_run_id == topic_run_id)
                ).scalar_one()
            print(
                f"Discovery topics in {DiscoveryTopic.__tablename__} "
                f"for this run: {discovery_topic_count}"
            )
        if inspector.has_table(WorkTopicAssignment.__tablename__):
            with engine.connect() as connection:
                outlier_assignment_count = connection.execute(
                    select(func.count())
                    .select_from(WorkTopicAssignment)
                    .where(
                        WorkTopicAssignment.topic_run_id == topic_run_id,
                        WorkTopicAssignment.is_outlier.is_(True),
                    )
                ).scalar_one()
                outlier_bertopic_count = connection.execute(
                    select(func.count())
                    .select_from(WorkTopicAssignment)
                    .where(
                        WorkTopicAssignment.topic_run_id == topic_run_id,
                        WorkTopicAssignment.bertopic_topic_id == -1,
                    )
                ).scalar_one()
            print(
                f"Outlier assignments (is_outlier=true): {outlier_assignment_count}"
            )
            print(
                f"Assignments with bertopic_topic_id=-1: {outlier_bertopic_count}"
            )
            if topic_run.get("outlier_count") is not None:
                print(
                    f"Run-row outlier_count: {topic_run['outlier_count']} "
                    f"(outlier_rate={topic_run.get('outlier_rate')})"
                )


Table queried: pwf_topic_runs
All topic runs in database: 1
Runs with exact window 2026-01-01..2026-01-31: 1
Runs covering the window: 1
Selected topic_run_id: fa3ddda5-8294-48fa-a581-1e8f45ba1ac7
Discovery topics in pwf_discovery_topics for this run: 280
Outlier assignments (is_outlier=true): 16427
Assignments with bertopic_topic_id=-1: 16427
Run-row outlier_count: 16427 (outlier_rate=0.42629885296102143)


,run_id,status,published_from,published_to,work_count,topic_count,outlier_count,outlier_rate,started_at,finished_at
0,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,succeeded,2026-01-01,2026-01-31,38534,280,16427,0.426299,2026-09-29 16:47:53.404489+00:00,2026-09-29 16:49:36.484277+00:00


## Summaries

Rows in `pwf_work_summaries` for the selected topic run, plus a short sample. Labeling may still be partial: summary count below the product of labeled topics × sample size means more topics await summarization.


In [6]:
print(f"Table queried: {WorkSummary.__tablename__}")
summary_count_for_run = 0
summary_topic_count = 0

if not inspector.has_table(WorkSummary.__tablename__):
    print("Table missing. Summaries: 0")
    display(
        pd.DataFrame(
            columns=["topic_run_id", "discovery_topic_id", "work_id", "summary_text"]
        )
    )
elif topic_run_id is None:
    with engine.connect() as connection:
        all_summaries = connection.execute(
            select(func.count()).select_from(WorkSummary)
        ).scalar_one()
    print(f"No covering topic run. All-rows count in table: {all_summaries}")
    print("Summaries for covering run: 0")
    display(
        pd.DataFrame(
            columns=["topic_run_id", "discovery_topic_id", "work_id", "summary_text"]
        )
    )
else:
    summary_count_stmt = (
        select(func.count())
        .select_from(WorkSummary)
        .where(WorkSummary.topic_run_id == topic_run_id)
    )
    summary_topics_stmt = (
        select(func.count(func.distinct(WorkSummary.discovery_topic_id)))
        .select_from(WorkSummary)
        .where(WorkSummary.topic_run_id == topic_run_id)
    )
    sample_stmt = (
        select(
            WorkSummary.topic_run_id,
            WorkSummary.discovery_topic_id,
            WorkSummary.work_id,
            WorkSummary.summary_text,
        )
        .where(WorkSummary.topic_run_id == topic_run_id)
        .limit(5)
    )
    with engine.connect() as connection:
        summary_count_for_run = connection.execute(summary_count_stmt).scalar_one()
        summary_topic_count = connection.execute(summary_topics_stmt).scalar_one()
        summary_sample = pd.DataFrame(connection.execute(sample_stmt).mappings().all())
    print(f"Summaries for topic_run_id={topic_run_id}: {summary_count_for_run}")
    print(f"Distinct discovery topics with ≥1 summary: {summary_topic_count}")
    display(summary_sample)


Table queried: pwf_work_summaries
Summaries for topic_run_id=fa3ddda5-8294-48fa-a581-1e8f45ba1ac7: 1686
Distinct discovery topics with ≥1 summary: 111


,topic_run_id,discovery_topic_id,work_id,summary_text
0,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,467ebed6-5ea0-4c55-bf3a-5d9258b88516,https://openalex.org/W7125780342,This study evaluates semaglutide's effects on mental health in individuals with pre-existing conditions. While the drug may offer certain psychological bene...
1,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,467ebed6-5ea0-4c55-bf3a-5d9258b88516,https://openalex.org/W7118831049,"Current research suggests that benzodiazepines, antidepressants, etifoxine, and afobazole may help manage severe adjustment disorder symptoms, though conclu..."
2,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,467ebed6-5ea0-4c55-bf3a-5d9258b88516,https://openalex.org/W7118084230,"This work explores ADHD treatment options for patients with hypertension, suggesting that non-stimulant medications like atomoxetine and certain SNRIs may p..."
3,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,467ebed6-5ea0-4c55-bf3a-5d9258b88516,https://openalex.org/W7125901965,"Exercise significantly reduces depressive symptoms and may outperform antidepressants in certain contexts, though conclusive evidence requires more direct c..."
4,fa3ddda5-8294-48fa-a581-1e8f45ba1ac7,467ebed6-5ea0-4c55-bf3a-5d9258b88516,https://openalex.org/W7124770632,This study examines whether non-aerobic exercise reduces depressive symptoms in adults with major depressive disorder (MDD) more effectively than standard m...


## Discovery topics, headlines, and labels

Discovery topics in `pwf_discovery_topics` joined to `pwf_topic_labels` for the selected run. Separate **succeeded** and **failed** label counts. Show a table of succeeded labels (id, size, headline, sample size, sampling method) and expand 2–3 succeeded examples (headline, concatenated summary, member summaries, sampled work titles).

Failed label rows have a null headline. Cause (ops): labeling API HTTP 403 `Project not found` — the API project is missing/unbound. Not a sampling bug (`sample_size` is populated on failed rows too).


In [7]:
print(f"Tables queried: {DiscoveryTopic.__tablename__}, {TopicLabel.__tablename__}")

topics_with_headline = 0
headline_rows = 0
label_status_counts = pd.DataFrame(columns=["status", "labels"])
labeled_topics = pd.DataFrame()
unlabeled_topic_count = 0
succeeded_label_count = 0
failed_label_count = 0
sample_size_vals = []

if not inspector.has_table(DiscoveryTopic.__tablename__):
    print(f"{DiscoveryTopic.__tablename__} missing. Discovery topics: 0")
elif topic_run_id is None:
    with engine.connect() as connection:
        discovery_topic_count = connection.execute(
            select(func.count()).select_from(DiscoveryTopic)
        ).scalar_one()
        if inspector.has_table(TopicLabel.__tablename__):
            headline_rows = connection.execute(
                select(func.count())
                .select_from(TopicLabel)
                .where(TopicLabel.headline.is_not(None))
            ).scalar_one()
    print("No covering topic run.")
    print(f"All discovery topics in database: {discovery_topic_count}")
    print(f"All non-null headlines in database: {headline_rows}")
else:
    topics_stmt = (
        select(
            DiscoveryTopic.discovery_topic_id,
            DiscoveryTopic.topic_run_id,
            DiscoveryTopic.size,
            TopicLabel.headline,
            TopicLabel.status,
            TopicLabel.sampling_method,
            TopicLabel.sample_size,
            TopicLabel.sampled_work_ids,
            TopicLabel.concatenated_summary_text,
        )
        .select_from(DiscoveryTopic)
        .outerjoin(
            TopicLabel,
            and_(
                TopicLabel.discovery_topic_id == DiscoveryTopic.discovery_topic_id,
                TopicLabel.topic_run_id == DiscoveryTopic.topic_run_id,
            ),
        )
        .where(DiscoveryTopic.topic_run_id == topic_run_id)
        .order_by(DiscoveryTopic.size.desc(), DiscoveryTopic.discovery_topic_id)
    )
    status_stmt = (
        select(TopicLabel.status, func.count().label("labels"))
        .where(TopicLabel.topic_run_id == topic_run_id)
        .group_by(TopicLabel.status)
        .order_by(TopicLabel.status)
    )
    with engine.connect() as connection:
        labels_frame = pd.DataFrame(connection.execute(topics_stmt).mappings().all())
        if inspector.has_table(TopicLabel.__tablename__):
            label_status_counts = pd.DataFrame(
                connection.execute(status_stmt).mappings().all()
            )
    discovery_topic_count = len(labels_frame)
    has_label_row = labels_frame["status"].notna() if len(labels_frame) else pd.Series(dtype=bool)
    succeeded_mask = (
        labels_frame["status"] == "succeeded" if len(labels_frame) else pd.Series(dtype=bool)
    )
    failed_mask = (
        labels_frame["status"] == "failed" if len(labels_frame) else pd.Series(dtype=bool)
    )
    succeeded_label_count = int(succeeded_mask.sum()) if len(labels_frame) else 0
    failed_label_count = int(failed_mask.sum()) if len(labels_frame) else 0
    topics_with_headline = (
        int(labels_frame.loc[succeeded_mask, "headline"].notna().sum())
        if len(labels_frame)
        else 0
    )
    headline_rows = topics_with_headline
    unlabeled_topic_count = int((~has_label_row).sum()) if len(labels_frame) else 0
    labeled_topics = (
        labels_frame.loc[succeeded_mask & labels_frame["headline"].notna()]
        .loc[:, ["discovery_topic_id", "size", "headline", "sample_size", "sampling_method", "status"]]
        .reset_index(drop=True)
    )
    failed_topics = (
        labels_frame.loc[failed_mask]
        .loc[:, ["discovery_topic_id", "size", "sample_size", "sampling_method", "status"]]
        .reset_index(drop=True)
    )
    sample_size_vals = (
        sorted(labels_frame["sample_size"].dropna().astype(int).unique().tolist())
        if len(labels_frame) and "sample_size" in labels_frame
        else []
    )

    print(f"Discovery topics for topic_run_id={topic_run_id}: {discovery_topic_count}")
    print(f"Label rows (any status): {int(has_label_row.sum()) if len(labels_frame) else 0}")
    print(f"Succeeded labels (with headline): {succeeded_label_count}")
    print(f"Failed labels (no headline): {failed_label_count}")
    print(
        f"{failed_label_count} topics have no headline because the labeling API "
        f"project is missing (HTTP 403 Project not found; ops confirmation)."
    )
    print(f"Topics still without a label row: {unlabeled_topic_count}")
    print(f"sample_size values on label rows: {sample_size_vals}")
    if discovery_topic_count:
        print(
            f"Labeling coverage (succeeded headlines / topics): "
            f"{topics_with_headline}/{discovery_topic_count}"
        )
    print("Label status counts:")
    display(label_status_counts if len(label_status_counts) else pd.DataFrame(columns=["status", "labels"]))
    print("Succeeded labeled topics (largest first):")
    display(labeled_topics)
    print(f"Failed labeled topics (no headline): {len(failed_topics)} (showing up to 5)")
    display(failed_topics.head(5))

    # Expanded examples: up to 3 largest succeeded labels
    example_n = min(3, len(labeled_topics))
    if example_n == 0:
        print("No succeeded headlines yet; expanded examples skipped.")
    else:
        print(f"\n## Expanded succeeded examples ({example_n})")
        example_ids = list(labeled_topics["discovery_topic_id"].head(example_n))
        for example_id in example_ids:
            row = labels_frame.loc[
                labels_frame["discovery_topic_id"] == example_id
            ].iloc[0]
            print("=" * 72)
            print(f"discovery_topic_id: {row['discovery_topic_id']}")
            print(f"size: {row['size']}  sample_size: {row['sample_size']}  "
                  f"sampling_method: {row['sampling_method']}  status: {row['status']}")
            print(f"headline: {row['headline']}")
            concat = row.get("concatenated_summary_text") or ""
            print("\nconcatenated_summary_text:")
            print(concat if concat else "(null)")

            member_stmt = (
                select(WorkSummary.work_id, WorkSummary.summary_text)
                .where(
                    WorkSummary.topic_run_id == topic_run_id,
                    WorkSummary.discovery_topic_id == example_id,
                )
                .order_by(WorkSummary.work_id)
            )
            with engine.connect() as connection:
                members = pd.DataFrame(connection.execute(member_stmt).mappings().all())
            print(f"\nmember summaries ({len(members)}):")
            if len(members):
                display(members)
            else:
                print("(none)")

            sampled_ids = list(row.get("sampled_work_ids") or [])
            title_rows = []
            if sampled_ids:
                payload = work_versions.c.payload
                title_stmt = (
                    select(
                        work_current.c.entity_id.label("work_id"),
                        payload["title"].astext.label("title"),
                    )
                    .select_from(
                        work_current.join(
                            work_versions,
                            work_versions.c.id == work_current.c.version_id,
                        )
                    )
                    .where(work_current.c.entity_id.in_(sampled_ids))
                )
                with engine.connect() as connection:
                    title_rows = pd.DataFrame(
                        connection.execute(title_stmt).mappings().all()
                    )
            print("sampled work titles:")
            display(title_rows if len(title_rows) else pd.DataFrame(columns=["work_id", "title"]))

print("---")
print(f"EVIDENCE selected_works={selected_works}")
print(f"EVIDENCE selected_with_vectors={selected_with_vectors}")
print(f"EVIDENCE collection={settings.qdrant_collection}")
print(f"EVIDENCE topic_run_id={topic_run_id}")
print(f"EVIDENCE topic_run_status={(topic_run or {}).get('status')}")
print(f"EVIDENCE discovery_topics={discovery_topic_count}")
print(f"EVIDENCE outlier_count_run={(topic_run or {}).get('outlier_count')}")
print(f"EVIDENCE outlier_assignments={outlier_assignment_count}")
print(f"EVIDENCE summaries={summary_count_for_run}")
print(f"EVIDENCE summary_topics={summary_topic_count}")
print(f"EVIDENCE label_rows={int(label_status_counts['labels'].sum()) if len(label_status_counts) else 0}")
print(f"EVIDENCE succeeded_labels={succeeded_label_count}")
print(f"EVIDENCE failed_labels={failed_label_count}")
print(f"EVIDENCE headlines={headline_rows}")
print(f"EVIDENCE unlabeled_topics={unlabeled_topic_count}")
print(f"EVIDENCE sample_size_values={sample_size_vals}")


Tables queried: pwf_discovery_topics, pwf_topic_labels
Discovery topics for topic_run_id=fa3ddda5-8294-48fa-a581-1e8f45ba1ac7: 280
Label rows (any status): 280
Succeeded labels (with headline): 108
Failed labels (no headline): 172
172 topics have no headline because the labeling API project is missing (HTTP 403 Project not found; ops confirmation).
Topics still without a label row: 0
sample_size values on label rows: [15]
Labeling coverage (succeeded headlines / topics): 108/280
Label status counts:
Succeeded labeled topics (largest first):
Failed labeled topics (no headline): 172 (showing up to 5)

## Expanded succeeded examples (3)
discovery_topic_id: 2f12a0f2-badc-4e2f-9462-1b9a2df6dabc
size: 588  sample_size: 15  sampling_method: representative_docs  status: succeeded
headline: Machine Learning for Neurological Diagnostics, Continuous Monitoring, and Rehabilitation

concatenated_summary_text:
This study introduces an ensemble learning framework based on Model Agnostic Meta Learning

,status,labels
0,failed,172
1,succeeded,108


,discovery_topic_id,size,headline,sample_size,sampling_method,status
0,2f12a0f2-badc-4e2f-9462-1b9a2df6dabc,588,"Machine Learning for Neurological Diagnostics, Continuous Monitoring, and Rehabilitation",15,representative_docs,succeeded
1,5264b5d8-009a-474e-ad98-ea5981a46d78,423,**Valorizing Agro-Food Byproducts through Advanced Processing for Sustainable Functional Foods**,15,representative_docs,succeeded
2,04809f3c-f023-473e-911e-563598605900,416,"Transcatheter and Surgical Structural Heart Interventions: Clinical Outcomes, Safety, and Therapeutic Innovations",15,representative_docs,succeeded
3,421d7235-fa58-43aa-846c-2f986f56db96,413,Overcoming Cancer Therapy Resistance Through Tumor Microenvironment Remodeling and Metabolic-Epigenetic Targeting,15,representative_docs,succeeded
4,4291e811-8f6f-4a15-9d0b-e3485760a123,292,**Advancing Crop Breeding for Abiotic Stress Tolerance and Yield Enhancement**,15,representative_docs,succeeded
...,...,...,...,...,...,...
103,3c33ce36-e44e-45ac-9837-1508ea1bf376,16,Prognostic Value of Clinical Risk Factors and Body Composition for Patient Outcomes,15,representative_docs,succeeded
104,4269161a-b4fa-4385-bea9-a945bab79580,16,"Sustainable Chemistry of Polymers and Bio-Based Materials: Curing, Coatings, and Safety",15,representative_docs,succeeded
105,4905f5c2-85e5-418c-b926-dfad01d1f1af,16,Perioperative Safety: Airway Management and Local Anesthetic Risk Factors,15,representative_docs,succeeded
106,12d36a60-8205-4deb-a6ad-51cbcc7a9765,15,"Cardiometabolic Risk, Chronic Disease Management, and Health Equity",15,representative_docs,succeeded


,discovery_topic_id,size,sample_size,sampling_method,status
0,5e5fef58-1e55-4760-9c23-3dbd64816088,402,15,representative_docs,failed
1,fcbbcd15-f9d3-49ef-836c-8ac116fd32af,364,15,representative_docs,failed
2,9291fca3-ff47-4183-a1a7-f29cda7b05ec,323,15,representative_docs,failed
3,a27930c6-61f9-432b-8ef3-f67f80213bf4,316,15,representative_docs,failed
4,fc2b885e-1181-4d7b-b43d-b1c3e9f1ccf3,304,15,representative_docs,failed


,work_id,summary_text
0,https://openalex.org/W7118184043,This exploratory randomized trial examined how structural-functional (SC-FC) brain changes correlate with post-stroke motor recovery following contralesiona...
1,https://openalex.org/W7119055152,This study integrates multi-scale EEG analysis with deep learning to predict individual responses to intermittent theta-burst stimulation (iTBS) in major de...
2,https://openalex.org/W7124178542,"This systematic review and meta-analysis of 44 randomized controlled trials (1,422 chronic stroke survivors) found that interactive cognitive-motor training..."
3,https://openalex.org/W7125523350,"This review examines the integration of deep learning with electroencephalography (EEG) for healthcare applications, focusing on neurological diagnostics, b..."
4,https://openalex.org/W7125645357,"This study introduces the Daily AHA Biomarker (DAB), a machine learning model that uses wrist-worn accelerometers to continuously monitor spontaneous upper ..."
5,https://openalex.org/W7125775060,This structured review of 21 studies (published ≥2015) evaluates machine and deep learning methods for predicting brain age from EEG data. Deep convolutiona...
6,https://openalex.org/W7127591640,This study introduces an ensemble learning framework based on Model Agnostic Meta Learning (MAML) to classify motor imagery (MI) tasks using electroencephal...
7,https://openalex.org/W7133236935,This study introduces a telerehabilitation system that assesses upper limb motor function in neurological injury patients using three inertial measurement u...
8,https://openalex.org/W7137310830,This study introduces a machine learning framework combining Principal Component Analysis (PCA) and K-Nearest Neighbors (KNN) to enhance early-stage Parkins...
9,https://openalex.org/W7138987781,"This study addresses data scarcity, class imbalance, and patient heterogeneity in assessing motor function for children with cerebral palsy (CP). The author..."


,work_id,title
0,https://openalex.org/W7118184043,Analysis of correlation between structural changes and functional response in post-stroke motor recovery after cM1 LF-rTMS: an exploratory randomized trial
1,https://openalex.org/W7119055152,Multi-scale EEG analysis identifies neural circuit signatures of iTBS responsiveness in major depressive disorder
2,https://openalex.org/W7124178542,"Effect of Interactive Cognitive-Motor Training on Balance, Mobility, and Cognition in Chronic Stroke Survivors"
3,https://openalex.org/W7125523350,Deep learning approaches for EEG-based healthcare applications: a comprehensive review
4,https://openalex.org/W7125645357,Machine Learning to Identify a New Digital Biomarker to Monitor Everyday Upper Limb Use in Children with Unilateral Cerebral Palsy
5,https://openalex.org/W7125775060,A Structured Review of EEG-Based Machine Learning Approaches for Brain Age Prediction
6,https://openalex.org/W7127591640,Model Agnostic Meta Learning Ensemble Based Prediction of Motor Imagery Tasks Using EEG Signals
7,https://openalex.org/W7133236935,Intelligent System for Upper Limb Motor Assessment Using Inertial Sensors and Machine Learning for Telerehabilitation Therapies
8,https://openalex.org/W7137310830,Advanced PCA-KNN Classification Technique for Parkinson’s disease Diagnosis at Early Stage
9,https://openalex.org/W7138987781,Diffusion-Augmented Spatiotemporal Graph Convolution for Clinical Gait and Motor Function Assessment


,work_id,summary_text
0,https://openalex.org/W7123910839,This study optimized the semi-solid-state fermentation (S-SSF) of black amaranth flour supplemented with soy okara using *Lactiplantibacillus pentosus* and ...
1,https://openalex.org/W7125659713,"This study developed functional cookies enriched with polyphenols extracted from habanero pepper leaves using a green, ultrasound-assisted natural deep eute..."
2,https://openalex.org/W7126048763,"This study presents a green, pulsed electric field–assisted extraction method optimized via response surface methodology to recover high-quality pectin from..."
3,https://openalex.org/W7126176351,This study investigates converting discarded persimmons into a prebiotic powder to modulate gut microbiota in postmenopausal women. After evaluating drying ...
4,https://openalex.org/W7127952049,"This study compares the effects of high-pressure processing (HPP at 400 and 600 MPa) and conventional thermal processing (CTP at 80°C, both for 10 minutes) ..."
5,https://openalex.org/W7128590295,This study develops and encapsulates bioactive peptide fractions derived from coconut cake waste to fortify yogurt. Utilizing enzymatic production and stabi...
6,https://openalex.org/W7131629125,This study optimized the production of moringa leaf vegetable leather using response surface methodology to determine the ideal glycerol concentration (2–4%...
7,https://openalex.org/W7140215505,"This study evaluated how cooking methods (boiling, baking, and raw) and plant parts (whole, flesh, skin) influence carotenoids, polyphenols, flavonoids, and..."
8,https://openalex.org/W7159929063,"This study optimized the extraction of ulvan polysaccharide from the Indonesian green macroalga *Ulva lactuca* and evaluated its chemical composition, struc..."
9,https://openalex.org/W7166509203,This study addresses the instability and inherent bitterness of botanical extracts in functional dairy by microencapsulating *Neolamarckia cadamba* (Kadam) ...


,work_id,title
0,https://openalex.org/W7123910839,Probiotic-Assisted Semi-Solid-State Fermentation of Amaranth Flour Using Response Surface Methodology for Enriched Bioactives
1,https://openalex.org/W7125659713,Simulated Digestion and Bioaccessibility of Cookies Enriched with Microencapsulated Polyphenols from Habanero Pepper Leaves Extracted Using NADES
2,https://openalex.org/W7126048763,Valorization of rambutan peel ( Nephelium lappaceum L.) byproducts through pulsed electric field-assisted green extraction of pectin optimized by response s...
3,https://openalex.org/W7126176351,Persimmon Powder from Discarded Fruits as a Potential Prebiotic to Modulate Gut Microbiota in Postmenopausal Women
4,https://openalex.org/W7127952049,"Effects of high-pressure processing and conventional thermal processing on enzyme activities, antioxidant properties, and volatile metabolic profiling of “ ..."
5,https://openalex.org/W7128590295,Development of stabilized coconut-waste peptides as bioactive ingredients to improve yogurt texture and functionality
6,https://openalex.org/W7131629125,Optimization of glycerol concentration and drying temperature on the characteristics of moringa leaf vegetable leather ( Moringa oleifera ) using response s...
7,https://openalex.org/W7140215505,"Comparative Assessment of Carotenoid and Polyphenol Contents, and Antioxidant Capacity in Thermally Processed Starchy Crops"
8,https://openalex.org/W7159929063,Ulvan Polysaccharide of Ulva lactuca From Coastal Indonesia: Functional Prebiotic Properties via Enzymatic and Nonenzymatic Extraction
9,https://openalex.org/W7166509203,Mechanistic Insights Into the Controlled Release and Antioxidant Protection of Neolamarckia cadamba Bioactives in Probiotic Yogurt: A Molecular and Microstr...


,work_id,summary_text
0,https://openalex.org/W7119469410,This case report details the successful percutaneous coronary intervention (PCI) of a 51-year-old male presenting with an inferior ST-segment elevation myoc...
1,https://openalex.org/W7122410424,This case report describes the successful transcatheter closure of a residual mid-leaflet mitral valve cleft in a 19-year-old patient with moderate mitral r...
2,https://openalex.org/W7123750183,"This single-center, randomized pilot study compared a novel 3D navigation-guided transcatheter aortic valve implantation (TAVI) technique with conventional ..."
3,https://openalex.org/W7124426819,*(Note: The provided text consists of a reference list rather than a traditional abstract. The summary below is synthesized from the title and the thematic ...
4,https://openalex.org/W7124832879,This retrospective study of 611 hypertrophic cardiomyopathy patients found that subcutaneous ICDs (S-ICDs) are associated with significantly fewer complicat...
5,https://openalex.org/W7125832480,"This meta-analysis of 29 studies evaluates the incidence, timing, and age-related factors of atrial fibrillation/flutter (AF/Af) following transcatheter pat..."
6,https://openalex.org/W7125925749,This case report describes the successful use of intravascular lithotripsy (IVL) to facilitate transcatheter aortic valve replacement (TAVR) in a 78-year-ol...
7,https://openalex.org/W7125927322,"The STAR-TEER trial is a multicenter, randomized, open-label study designed to establish evidence-based antithrombotic strategies following transcatheter ed..."
8,https://openalex.org/W7126223452,This review examines the evolution and expanding clinical applications of percutaneous left atrial appendage closure (LAAC) for stroke prevention in non-val...
9,https://openalex.org/W7129486267,"This report presents preliminary 5-year outcomes from the international IMPACT registry, evaluating a novel bioprosthetic valve used in surgical aortic valv..."


,work_id,title
0,https://openalex.org/W7122410424,Transcatheter Closure of a Residual Mid‐Leaflet Mitral Valve Cleft With a Siege Vascular Plug
1,https://openalex.org/W7123750183,Evaluation of the Efficacy and Safety of a 3D Navigation-Guided Transcatheter Aortic Valve Implantation Technique in Patients With Severe Aortic Stenosis an...
2,https://openalex.org/W7124426819,Perioperative Management of An Elderly with Coronary Artery Disease and Aortic Stenosis Undergoing Emergency ORIF for Intertrochanteric Fracture. Report of ...
3,https://openalex.org/W7124832879,S‐ICD in Hypertrophic Cardiomyopathy: Fewer Complications Without Increased Inappropriate Shocks Versus Transvenous Systems
4,https://openalex.org/W7125832480,Incidence of Atrial Fibrillation Post Patent Foramen Ovale (PFO) Closure: A Meta-analysis
5,https://openalex.org/W7125925749,Lithotripsy-assisted TAVR for severe calcific aortic stenosis. Case report
6,https://openalex.org/W7125927322,STrategies for antithrombotic tRreatment following transcatheter edge-to-edge repair in patients with severe mitral regurgitation: Rationale and design of S...
7,https://openalex.org/W7126223452,Percutaneous left atrial appendage closure: evolution of devices and expanding clinical applications
8,https://openalex.org/W7129486267,5-Year Outcomes from the IMPACT Registry: Early Look at Preliminary Closing Phase Data on Bioprosthetic Surgical Aortic Valve Replacement
9,https://openalex.org/W7129557250,Minimal Invasive Aortic Valve Replacement with Aortic Annular Enlargement


## Weak-signal candidate read

Judgment below uses only **succeeded** headlines and the counts printed above (no invented metrics). Failed topics have no headline because the labeling API project is missing.


In [8]:
from IPython.display import Markdown

lines = []
if topic_run_id is None:
    lines.append("No covering topic run; cannot judge labeled discovery topics yet.")
else:
    lines.append(
        f"Succeeded headlines: {succeeded_label_count}/{discovery_topic_count}. "
        f"Failed (no headline): {failed_label_count} — labeling API project missing "
        f"(HTTP 403 Project not found). "
        f"Work summaries: {summary_count_for_run} across {summary_topic_count} topics. "
        f"sample_size on label rows: {sample_size_vals}."
    )
    oc = (topic_run or {}).get("outlier_count")
    orate = (topic_run or {}).get("outlier_rate")
    if oc is not None:
        lines.append(
            f"Outlier share high: outlier_count={oc} (outlier_rate={orate}); "
            f"assignment table {outlier_assignment_count} is_outlier rows."
        )
    if len(labeled_topics):
        sizes = labeled_topics["size"].astype(int)
        samples = labeled_topics["sample_size"].astype(int)
        advancing = int(
            labeled_topics["headline"].astype(str).str.match(r"(?i)^advancing\b").sum()
        )
        lines.append(
            f"Succeeded topic sizes {sizes.min()}–{sizes.max()} "
            f"(median {int(sizes.median())}); sample_size values "
            f"{sorted(samples.unique().tolist())} (target min(15, topic_size))."
        )
        example_heads = list(labeled_topics["headline"].head(3))
        lines.append(
            "Largest succeeded headlines: "
            + "; ".join(f"“{h}”" for h in example_heads)
            + "."
        )
        lines.append(
            f"Of {len(labeled_topics)} succeeded headlines, {advancing} start with "
            f"“Advancing” (generic cue). From expanded examples: when member summaries "
            f"and sampled titles name the same concrete method/material as the headline, "
            f"treat as a usable weak-signal *candidate*; when the headline stays broad "
            f"while members span loosely related work, treat as a coarse bucket. "
            f"{failed_label_count} topics have no headline because the labeling API "
            f"project is missing — do not judge those as unlabeled-by-sampling."
        )
    else:
        lines.append(
            f"No succeeded headlines; {failed_label_count} failed for missing API project. "
            f"Weak-signal candidate read deferred."
        )

display(Markdown("\n\n".join(lines)))


Succeeded headlines: 108/280. Failed (no headline): 172 — labeling API project missing (HTTP 403 Project not found). Work summaries: 1686 across 111 topics. sample_size on label rows: [15].

Outlier share high: outlier_count=16427 (outlier_rate=0.42629885296102143); assignment table 16427 is_outlier rows.

Succeeded topic sizes 15–588 (median 54); sample_size values [15] (target min(15, topic_size)).

Largest succeeded headlines: “Machine Learning for Neurological Diagnostics, Continuous Monitoring, and Rehabilitation”; “**Valorizing Agro-Food Byproducts through Advanced Processing for Sustainable Functional Foods**”; “Transcatheter and Surgical Structural Heart Interventions: Clinical Outcomes, Safety, and Therapeutic Innovations”.

Of 108 succeeded headlines, 6 start with “Advancing” (generic cue). From expanded examples: when member summaries and sampled titles name the same concrete method/material as the headline, treat as a usable weak-signal *candidate*; when the headline stays broad while members span loosely related work, treat as a coarse bucket. 172 topics have no headline because the labeling API project is missing — do not judge those as unlabeled-by-sampling.